# 03 · Reranking
### Practical LLM Engineering — Module 04: RAG Systems

> **What you'll learn**
> - Why bi-encoder retrieval needs a reranking stage
> - Cross-encoder reranking: architecture and scoring
> - LLM-as-reranker: listwise and pointwise approaches
> - ColBERT-style late interaction scoring
> - Measuring reranking impact: NDCG@k improvement
> - Engineering: latency budget and candidate pool sizing

---


## 1. Overview

**The recall-precision gap:**

```
Bi-encoder (retrieval):    Fast   | Approximate | Returns top-100
Cross-encoder (reranker):  Slow   | Precise     | Reranks top-100 → top-5

Why the gap?
  Bi-encoder  embeds query and document INDEPENDENTLY → misses fine-grained interaction
  Cross-encoder encodes [query + document] JOINTLY   → full attention interaction
```

### Reranking methods

| Method | Accuracy | Speed | Cost |
|---|---|---|---|
| Score threshold filter | ★ | ⚡⚡⚡ | Free |
| Cross-encoder (BERT-based) | ★★★★ | ⚡⚡ | Free (local) |
| LLM pointwise | ★★★★ | ⚡ | API cost |
| LLM listwise | ★★★★★ | ⚡ | API cost |
| ColBERT (late interaction) | ★★★★★ | ⚡⚡ | Free (local) |


## 2. Setup

In [ ]:
# !pip install numpy matplotlib scikit-learn -q
# ── Shared utilities (carried from Module 03) ────────────────────────
import os, re, json, time, math, random, textwrap
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional
from collections import defaultdict
from sklearn.preprocessing import normalize

# ── Mock embedder ─────────────────────────────────────────────────────
class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim   = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        tokens = text.lower().split()
        if not tokens: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(t)] for t in tokens], axis=0)
        return v / max(np.linalg.norm(v), 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts]))

# ── BM25 ──────────────────────────────────────────────────────────────
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self._corpus, self._doc_freqs, self._avgdl, self._N = [], defaultdict(int), 0.0, 0
    def _tok(self, t): return re.findall(r"\b\w+\b", t.lower())
    def fit(self, docs):
        self._corpus = [self._tok(d) for d in docs]; self._N = len(self._corpus)
        self._avgdl  = np.mean([len(d) for d in self._corpus])
        for doc in self._corpus:
            for t in set(doc): self._doc_freqs[t] += 1
    def _idf(self, t):
        n = self._doc_freqs.get(t, 0)
        return math.log((self._N - n + 0.5) / (n + 0.5) + 1)
    def get_scores(self, query):
        qt = self._tok(query)
        scores = []
        for doc in self._corpus:
            dl = len(doc); tf = defaultdict(int)
            for t in doc: tf[t] += 1
            s = sum(self._idf(t) * tf[t] * (self.k1+1) /
                    (tf[t] + self.k1*(1-self.b+self.b*dl/self._avgdl))
                    for t in qt if t in tf)
            scores.append(s)
        return np.array(scores)
    def get_top_k(self, query, k=5):
        s = self.get_scores(query); idx = np.argsort(-s)[:k]
        return [(int(i), float(s[i])) for i in idx]

# ── Vector store ──────────────────────────────────────────────────────
@dataclass
class Doc:
    id: str; text: str; metadata: dict = field(default_factory=dict)

class VectorStore:
    def __init__(self, embedder, dim=64):
        self.embedder = embedder
        self._vecs: np.ndarray = np.zeros((0, dim), dtype=np.float32)
        self._docs: list[Doc]  = []
    def add(self, docs: list[Doc]):
        vecs = self.embedder.embed([d.text for d in docs])
        self._vecs = np.vstack([self._vecs, vecs]) if len(self._vecs) else vecs
        self._docs.extend(docs)
    def search(self, query: str, k=5, where=None) -> list[tuple[Doc, float]]:
        if not self._docs: return []
        q = self.embedder.embed([query])[0]
        scores = self._vecs @ q
        if where:
            for i, doc in enumerate(self._docs):
                for key, val in where.items():
                    if doc.metadata.get(key) != val: scores[i] = -999
        top = np.argsort(-scores)[:k]
        return [(self._docs[i], float(scores[i])) for i in top]
    def __len__(self): return len(self._docs)

# ── Mock LLM ──────────────────────────────────────────────────────────
@dataclass
class LLMResponse:
    text: str; input_tokens: int; output_tokens: int; latency_ms: float
    @property
    def total_tokens(self): return self.input_tokens + self.output_tokens

class MockLLM:
    """Returns templated answers grounded in provided context."""
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def complete(self, system: str, user: str,
                 max_tokens=512, temperature=0.0) -> LLMResponse:
        time.sleep(0.02)
        ctx_match = re.search(r"Context:(.*?)(?:Question:|$)", user, re.DOTALL)
        ctx = ctx_match.group(1).strip()[:200] if ctx_match else ""
        q   = re.search(r"Question:(.*?)$", user, re.DOTALL)
        q   = q.group(1).strip()[:80] if q else user[:80]
        answer = (f"Based on the provided context, {q.lower().rstrip('?')} "
                  f"can be understood as follows: {ctx[:120]}... "
                  f"This answer is grounded in the retrieved documents.")
        n_in  = len((system+user).split())
        n_out = len(answer.split())
        return LLMResponse(answer, n_in, n_out, 45.0)
    def __call__(self, system, user, **kw): return self.complete(system, user, **kw)

embedder = MockEmbedder(dim=64)
llm      = MockLLM()
print("✅ Shared utilities loaded (embedder + BM25 + VectorStore + MockLLM)")


In [ ]:
CORPUS = [
    Doc("d01", "Self-attention computes Q·K^T/sqrt(d_k) scores, applies softmax, then weights V.", {"rel_q1": 5}),
    Doc("d02", "Multi-head attention runs attention in parallel across h heads then concatenates.", {"rel_q1": 4}),
    Doc("d03", "BERT uses bidirectional attention; GPT uses causal (left-to-right) attention.",    {"rel_q1": 3}),
    Doc("d04", "BM25 is a sparse bag-of-words retrieval model with TF-IDF and length normalisation.", {"rel_q1": 1}),
    Doc("d05", "FAISS supports GPU-accelerated billion-scale similarity search.",                  {"rel_q1": 1}),
    Doc("d06", "Gradient descent minimises loss by moving weights in the negative gradient direction.", {"rel_q1": 0}),
    Doc("d07", "Attention in transformers allows each token to attend to all other tokens.",       {"rel_q1": 5}),
    Doc("d08", "The softmax function converts logits to a probability distribution over tokens.", {"rel_q1": 2}),
    Doc("d09", "Layer normalisation stabilises training by normalising activations per layer.",    {"rel_q1": 1}),
    Doc("d10", "Positional encodings inject sequence position information into token embeddings.", {"rel_q1": 2}),
]
store = VectorStore(embedder); store.add(CORPUS)
QUERY1 = "how does attention mechanism work in transformers"
print(f"Corpus: {len(CORPUS)} docs | Query: {QUERY1}")


## 3. Cross-Encoder Reranking

A cross-encoder takes the concatenated [CLS] query [SEP] document [SEP] as input and outputs a scalar relevance score. Full bi-directional attention between query and document tokens enables fine-grained matching.

$$
\text{score}(q, d) = \text{MLP}(h_{\text{CLS}}) \in \mathbb{R}
$$

where $h_{\text{CLS}}$ is the transformer's CLS representation of the concatenated input.


In [ ]:
class CrossEncoderReranker:
    """
    Cross-encoder reranker (mock implementation).
    Real usage:
        from sentence_transformers import CrossEncoder
        model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        scores = model.predict([(query, doc) for doc in docs])
    """
    def __init__(self, embedder):
        self.embedder = embedder

    def score(self, query: str, doc_text: str) -> float:
        """Score a (query, document) pair — higher is more relevant."""
        q_tok = set(query.lower().split())
        d_tok = set(doc_text.lower().split())
        # Keyword overlap
        overlap = len(q_tok & d_tok) / max(len(q_tok), 1)
        # Embedding similarity
        vecs = self.embedder.embed([query, doc_text])
        emb_sim = float(np.dot(vecs[0], vecs[1]))
        return 0.35 * overlap + 0.65 * emb_sim

    def rerank(self, query: str, candidates: list[tuple[Doc, float]],
               top_k: int = 5) -> list[tuple[Doc, float]]:
        scored = [(doc, self.score(query, doc.text)) for doc, _ in candidates]
        return sorted(scored, key=lambda x: -x[1])[:top_k]


# ── Measure NDCG@k improvement ───────────────────────────────────────
def ndcg_at_k(retrieved_ids: list[str], relevance_map: dict[str, int], k: int) -> float:
    dcg   = sum(relevance_map.get(did, 0) / math.log2(r+2)
                for r, did in enumerate(retrieved_ids[:k]))
    ideal = sorted(relevance_map.values(), reverse=True)[:k]
    idcg  = sum(rel / math.log2(r+2) for r, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

relevance_map = {doc.id: doc.metadata["rel_q1"] for doc in CORPUS}
cross_enc     = CrossEncoderReranker(embedder)

# Before reranking (bi-encoder only)
raw_retrieval = store.search(QUERY1, k=10)
raw_ids       = [doc.id for doc, _ in raw_retrieval]

# After reranking
reranked      = cross_enc.rerank(QUERY1, raw_retrieval, top_k=5)
reranked_ids  = [doc.id for doc, _ in reranked]

print(f"Query: {QUERY1}\n")
print(f"{'k':>4} {'NDCG (bi-enc)':>15} {'NDCG (reranked)':>17} {'Δ':>8}")
print("-" * 48)
for k in [1, 3, 5]:
    nd_raw  = ndcg_at_k(raw_ids, relevance_map, k)
    nd_rer  = ndcg_at_k(reranked_ids, relevance_map, k)
    print(f"{k:>4} {nd_raw:>15.3f} {nd_rer:>17.3f} {nd_rer-nd_raw:>+8.3f}")

print()
print("Top-5 after reranking:")
for rank, (doc, score) in enumerate(reranked, 1):
    rel = relevance_map.get(doc.id, 0)
    print(f"  #{rank} [{doc.id}] score={score:.3f} rel={rel} {doc.text[:55]}")


## 4. LLM-as-Reranker

In [ ]:
# ── Pointwise LLM reranker ───────────────────────────────────────────
def llm_pointwise_rerank(query: str, candidates: list[tuple[Doc, float]],
                          llm: MockLLM, top_k: int = 5) -> list[tuple[Doc, float]]:
    """Score each candidate independently with the LLM."""
    system = """You are a relevance scoring assistant.
Score the relevance of a document to a query on a scale of 1-10.
Respond with only a single integer."""
    scored = []
    for doc, _ in candidates:
        user = f"Query: {query}\nDocument: {doc.text}\nRelevance (1-10):"
        resp = llm(system, user, max_tokens=5)
        # Extract number from response
        nums = re.findall(r"\b([1-9]|10)\b", resp.text)
        score = float(nums[0]) / 10.0 if nums else 0.5
        scored.append((doc, score))
    return sorted(scored, key=lambda x: -x[1])[:top_k]


# ── Listwise LLM reranker ────────────────────────────────────────────
def llm_listwise_rerank(query: str, candidates: list[tuple[Doc, float]],
                         llm: MockLLM, top_k: int = 5) -> list[tuple[Doc, float]]:
    """Rank all candidates simultaneously in a single LLM call."""
    doc_list = "\n".join(f"[{i+1}] {doc.text[:80]}"
                           for i, (doc, _) in enumerate(candidates))
    system   = "You are a document ranking assistant."
    user     = (f"Query: {query}\n\nDocuments:\n{doc_list}\n\n"
                f"Rank these documents from most to least relevant to the query. "
                f"Output only the document numbers in order, comma-separated.")
    resp     = llm(system, user, max_tokens=50)

    # Parse ranking
    order = [int(x)-1 for x in re.findall(r"\d+", resp.text)
             if 0 < int(x) <= len(candidates)]
    # Fill in any missed docs
    all_pos = list(range(len(candidates)))
    for p in all_pos:
        if p not in order: order.append(p)

    return [(candidates[i][0], 1.0/(r+1)) for r, i in enumerate(order)][:top_k]


# Compare all three rerankers
print("Reranker comparison — NDCG@5\n")
candidates = store.search(QUERY1, k=10)

methods = {
    "Bi-encoder only":     [(doc, sc) for doc, sc in candidates[:5]],
    "Cross-encoder":       cross_enc.rerank(QUERY1, candidates, 5),
    "LLM pointwise":       llm_pointwise_rerank(QUERY1, candidates, llm, 5),
    "LLM listwise":        llm_listwise_rerank(QUERY1, candidates, llm, 5),
}

print(f"{'Method':<20} {'NDCG@3':>8} {'NDCG@5':>8}")
print("-" * 38)
for name, results in methods.items():
    ids = [doc.id for doc, _ in results]
    n3  = ndcg_at_k(ids, relevance_map, 3)
    n5  = ndcg_at_k(ids, relevance_map, 5)
    print(f"{name:<20} {n3:>8.3f} {n5:>8.3f}")


## 5. ColBERT-Style Late Interaction

ColBERT (Khattab & Zaharia, 2020) encodes query and document token-by-token, then scores via **MaxSim**:

$$
\text{score}(q, d) = \sum_{i \in q} \max_{j \in d} E_{q_i} \cdot E_{d_j}
$$

Each query token finds its best-matching document token. This is more expressive than a single CLS vector but cheaper than a full cross-encoder (document vectors can be pre-computed).


In [ ]:
class ColBERTStyleScorer:
    """
    Mock ColBERT-style late interaction.
    Real: pip install colbert-ai
    """
    def __init__(self, embedder):
        self.embedder = embedder
        self._dim     = embedder.dim

    def _token_vecs(self, text: str, max_tokens: int = 16) -> np.ndarray:
        """Produce per-token embeddings (simulated with word perturbations)."""
        words = text.lower().split()[:max_tokens]
        vecs  = []
        for i, w in enumerate(words):
            v    = self.embedder.embed_one(w)
            # Add small position perturbation
            v   += np.random.RandomState(hash(w+str(i)) % 2**31).randn(self._dim) * 0.05
            v   /= max(np.linalg.norm(v), 1e-9)
            vecs.append(v)
        return np.array(vecs, dtype=np.float32) if vecs else np.zeros((1, self._dim))

    def score(self, query: str, doc_text: str) -> float:
        q_vecs = self._token_vecs(query)
        d_vecs = self._token_vecs(doc_text, max_tokens=64)
        # MaxSim: for each query token, find max similarity to any doc token
        sims   = q_vecs @ d_vecs.T       # (|q|, |d|)
        maxsim = sims.max(axis=1)        # (|q|,)
        return float(maxsim.mean())

    def rerank(self, query: str, candidates: list[tuple[Doc, float]],
               top_k: int = 5) -> list[tuple[Doc, float]]:
        scored = [(doc, self.score(query, doc.text)) for doc, _ in candidates]
        return sorted(scored, key=lambda x: -x[1])[:top_k]


colbert = ColBERTStyleScorer(embedder)
colbert_results = colbert.rerank(QUERY1, candidates, 5)
ids_col = [doc.id for doc, _ in colbert_results]
n5_col  = ndcg_at_k(ids_col, relevance_map, 5)
print(f"ColBERT-style NDCG@5: {n5_col:.3f}")

print("\nColBERT top-5:")
for rank, (doc, score) in enumerate(colbert_results, 1):
    rel = relevance_map.get(doc.id, 0)
    print(f"  #{rank} [{doc.id}] maxsim={score:.3f} rel={rel}  {doc.text[:55]}")


## 6. Candidate Pool Size vs Reranking Quality

In [ ]:
# ── Recall@5 as a function of candidate pool size ────────────────────
relevant_ids = {doc.id for doc in CORPUS if doc.metadata["rel_q1"] >= 4}
pool_sizes   = [3, 5, 8, 10]

print(f"Relevant docs (rel≥4): {relevant_ids}\n")
print(f"{'Pool size':>10} {'Recall (bi-enc)':>17} {'Recall (cross-enc)':>20}")
print("-" * 52)

for n in pool_sizes:
    pool      = store.search(QUERY1, k=n)
    bi_ids    = {doc.id for doc, _ in pool[:5]}
    reranked  = cross_enc.rerank(QUERY1, pool, top_k=5)
    rer_ids   = {doc.id for doc, _ in reranked}
    rec_bi    = len(bi_ids  & relevant_ids) / len(relevant_ids)
    rec_rer   = len(rer_ids & relevant_ids) / len(relevant_ids)
    print(f"{n:>10} {rec_bi:>17.3f} {rec_rer:>20.3f}")

print("\n💡 Larger pool → reranker has more candidates to promote relevant docs.")


## 7. Key Takeaways

| Concept | Summary |
|---|---|
| **Recall-precision gap** | Bi-encoder fast but approximate; cross-encoder precise but slow |
| **Two-stage pipeline** | Stage 1: bi-encoder recall top-100; Stage 2: rerank to top-5 |
| **Cross-encoder** | Full joint attention; best local reranker |
| **LLM pointwise** | Score each doc independently; flexible rubric |
| **LLM listwise** | Score all docs in one call; cheaper but less controllable |
| **ColBERT MaxSim** | Per-token matching; more expressive than CLS; pre-computable |
| **Pool sizing** | Larger first-stage pool gives reranker more to work with |

---
**Next notebook →** `04_rag_evaluation.ipynb`
